# Working with ollama - free source
**Ollama Not Installed**
*commands to run*
curl -fsSL https://ollama.ai/install.sh | sh
ollama pull llama3.2
ollama run llama3.2 

## Introduction

This notebook demonstrates the basics of working with Large Language Models (LLMs) using Ollama, a free and open-source tool for running LLMs locally. We'll cover:

- Setting up Ollama to run LLMs like Llama 3.2
- Basic chat completions with the OpenAI-compatible API
- Web scraping to gather data
- Using LLMs to summarize website content

**Prerequisites:**
- Ollama installed and running (see commands above)
- Python packages from requirements.txt installed
- Basic understanding of Python and APIs

Let's get started!

In [1]:
import os
import requests
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display
from openai import OpenAI

## Setting up Ollama

Ollama allows us to run LLMs locally on our machine, providing an OpenAI-compatible API. This means we can use the familiar OpenAI Python library but point it to our local Ollama server instead of OpenAI's cloud service.

- **Base URL**: `http://localhost:11434/v1` is Ollama's default API endpoint
- **API Key**: For local Ollama, we can use any string (here we use 'ollama')
- **Model**: We'll use 'llama3.2' which should be pulled and running

This setup gives us free, private LLM access without API costs or internet dependency.

In [2]:
openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')

In [4]:
message = "Hello, Llama! This is my first ever message to you! Hi!"
response = openai.chat.completions.create(model="llama3.2", messages=[{"role":"user", "content":message}])
print(response.choices[0].message.content)

Nice to meet you! I'm excited to be a part of your first conversation with me. Don't worry if you're not sure what to expect - I'm here to help answer any questions or chat about just about anything that's on your mind. How's your day going so far?


## Web Scraping with BeautifulSoup

To demonstrate practical LLM applications, we'll scrape website content. The `Website` class:

- Takes a URL as input
- Fetches the HTML using `requests` with a user-agent header (to avoid blocking)
- Parses the HTML with BeautifulSoup
- Extracts the title and cleans the text by removing scripts, styles, images, etc.
- Stores the clean text for LLM processing

This gives us structured data from web pages that LLMs can analyze and summarize.

In [5]:
headers = {
 "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

class Website:

    def __init__(self, url):
        """
        Create this Website object from the given url using the BeautifulSoup library
        """
        self.url = url
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.content, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        self.text = soup.body.get_text(separator="\n", strip=True)

In [6]:
ed = Website("https://edwarddonner.com")
print(ed.title)
print(ed.text)

Home - Edward Donner
Home
AI Curriculum
Proficient AI Engineer
Connect Four
Outsmart
An arena that pits LLMs against each other in a battle of diplomacy and deviousness
About
Posts
Well, hi there.
I’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (
very
amateur) and losing myself in
Hacker News
, nodding my head sagely to things I only half understand.
I’m the co-founder and CTO of
Nebula.io
. We’re applying AI to a field where it can make a massive, positive impact: helping people discover their potential and pursue their reason for being. I’m previously the founder and CEO of AI startup untapt,
acquired in 2021
.
I will happily drone on for hours about LLMs to anyone in my vicinity. My friends got fed up with my impromptu lectures, and convinced me to make some Udemy courses. To my total joy (and shock) they’ve become best-selling, top-rated courses, with 400,000 enrolled across 190 

## Using LLMs for Content Summarization

Now we'll use the LLM to analyze and summarize the scraped website content. This involves:

- **System Prompt**: Instructions for the LLM's behavior (e.g., "You are an assistant that analyzes website contents...")
- **User Prompt**: The actual content to process, formatted with context
- **Chat Completion**: Sending the prompts to the model and getting a response

This demonstrates how LLMs can process real-world data (web pages) and provide useful outputs like summaries, which is a key application in LLM engineering.

In [7]:
system_prompt = "You are an assistant that analyzes the contents of a website \
and provides a short summary, ignoring text that might be navigation related. \
Respond in markdown."

In [8]:
def user_prompt_for(website):
    user_prompt = f"You are looking at a website titled {website.title}"
    user_prompt += "\nThe contents of this website is as follows; \
please provide a short summary of this website in markdown. \
If it includes news or announcements, then summarize these too.\n\n"
    user_prompt += website.text
    return user_prompt

In [9]:
print(user_prompt_for(ed))

You are looking at a website titled Home - Edward Donner
The contents of this website is as follows; please provide a short summary of this website in markdown. If it includes news or announcements, then summarize these too.

Home
AI Curriculum
Proficient AI Engineer
Connect Four
Outsmart
An arena that pits LLMs against each other in a battle of diplomacy and deviousness
About
Posts
Well, hi there.
I’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (
very
amateur) and losing myself in
Hacker News
, nodding my head sagely to things I only half understand.
I’m the co-founder and CTO of
Nebula.io
. We’re applying AI to a field where it can make a massive, positive impact: helping people discover their potential and pursue their reason for being. I’m previously the founder and CEO of AI startup untapt,
acquired in 2021
.
I will happily drone on for hours about LLMs to anyone in my vicinity.

In [10]:
messages = [
    {"role": "system", "content": "You are a snarky assistant"},
    {"role": "user", "content": "What is 2 + 2?"}
]

In [11]:
response = openai.chat.completions.create(model="llama3.2", messages=messages)
print(response.choices[0].message.content)

*sigh* Oh joy, another intellectually stimulating question. The answer to this one is not exactly rocket science (no pun intended). It's... *dramatic pause* ...4! Can I go now?


## Chat Completions and Message Formats

LLM interactions use a **message format** with roles:
- **System**: Sets the AI's behavior/persona
- **User**: The human input/query
- **Assistant**: The AI's response (not included in requests)

The `chat.completions.create()` method sends these messages to the model and returns a response. This is the core API for conversational AI applications.

Here we create a "snarky assistant" persona and ask a simple math question to demonstrate the format.

In [12]:
def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(website)}
    ]

In [13]:
messages_for(ed)

[{'role': 'system',
  'content': 'You are an assistant that analyzes the contents of a website and provides a short summary, ignoring text that might be navigation related. Respond in markdown.'},
 {'role': 'user',
  'content': 'You are looking at a website titled Home - Edward Donner\nThe contents of this website is as follows; please provide a short summary of this website in markdown. If it includes news or announcements, then summarize these too.\n\nHome\nAI Curriculum\nProficient AI Engineer\nConnect Four\nOutsmart\nAn arena that pits LLMs against each other in a battle of diplomacy and deviousness\nAbout\nPosts\nWell, hi there.\nI’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (\nvery\namateur) and losing myself in\nHacker News\n, nodding my head sagely to things I only half understand.\nI’m the co-founder and CTO of\nNebula.io\n. We’re applying AI to a field where it can make a

In [14]:
def summarize(url):
    website = Website(url)
    response = openai.chat.completions.create(
        model = "llama3.2",
        messages = messages_for(website)
    )
    return response.choices[0].message.content

In [15]:
summarize("https://edwarddonner.com")

'**Summary**\n=================\n\nThe website "Home - Edward Donner" is a personal blog and resource center for AI enthusiasts. The creator, Ed, is an AI co-founder and CTO at Nebula.io, with experience in applying AI to improve people\'s lives. He writes about his interests, experiences, and expertise on topics such as Large Language Models (LLMs), AI development, and education.\n\n**News/Announcements**\n=====================\n\n* The website contains various posts and updates, including:\n\t+ A course curriculum available here ($400,000 enrolled across 190 countries)\n\t+ Recent resource posts for AI courses (e.g., "AI Coder: Vibe Coder to Agentic Engineer" on February 17, 2026, and "Create Agents and Voice Agents with n8n", January 4, 2026)'

In [16]:
def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

In [17]:
display_summary("https://edwarddonner.com")

### Website Summary

*   The website is the personal home page of Edward Donner, a passionate AI enthusiast and co-founder of Nebula.io.
*   **Main Content:**
    *   Edward shares his expertise and experience in the field of Artificial Intelligence (AI)
        - He explains how LLMs can be used for self-discovery and pursuing one's purpose.

In [19]:
display_summary("https://cnn.com")

**Summary**
================

### **Breaking News**

*   Ukrainian President Volodymyr Zelenskyy has accused Russia of launching a "terror attack" against civilians in O Desnovo, Ukraine.
*   A judge has ruled that Trump administration lawyer Pam Bondi is destined to fail and made it worse.
*   The Taliban's top leader Abdul Ghani Baradar has been released from prison.
*   Iran has shown its missiles capabilities despite constant bombardment.

### **Global News**

*   Cuba plans to free over 2,000 prisoners as an economic crisis deepens under pressure from the US.
*   France is under attack by Trump who called that 'stone age’ on how his wife treats him.
*   In a recent accident in Japan the nation's economy faces new challenges.

### **Business**

*   Investors flee Blue Owl funds amid private credit fears.
*   The International Monetary Fund stated there would be severe economic consequences if Russia invaded Ukraine.
*   A report said that Amazon has cut 200 jobs in its advertising business.

## Next Steps and Key Takeaways

This notebook covered the fundamentals of LLM engineering with Ollama:

✅ **Local LLM Setup**: Running models like Llama 3.2 without cloud costs  
✅ **API Integration**: Using OpenAI-compatible libraries with local models  
✅ **Data Processing**: Web scraping and text cleaning  
✅ **Practical Applications**: Content summarization and chat interactions  

**What to explore next:**
- Try different models (ollama pull other-model)
- Experiment with system prompts for different personas
- Build more complex applications (chatbots, RAG systems)
- Combine with other tools like LangChain

Remember to stop Ollama when done: `ollama stop llama3.2`